<a href="https://colab.research.google.com/github/Aniket-034/APS-LAB-/blob/main/Lab_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Travelling Salesman Problem using Dynamic Programming.
Problem Description:
Given a set of cities and distances, find the shortest possible route that visits each city once and returns to the starting city.

#Theory
TSP uses dynamic programming with bit masking. Store results of visited subsets to reduce repeated work.

## Algorithm
1. Start from source city
2. Mark visited cities using bit mask
3. Try all unvisited cities
4. Choose minimum cost path
5. Return to source city

In [5]:
def tsp(graph):
    n = len(graph)
    # dp[city][mask] stores (minimum_cost, next_city_in_optimal_path)
    # Initialize with (infinity, -1) to indicate uncomputed states
    dp = [[(float('inf'), -1)] * (1 << n) for _ in range(n)]

    def solve(city, mask):
        # Base case: if all cities have been visited
        if mask == (1 << n) - 1:
            # Cost to return to the starting city (city 0)
            return graph[city][0], 0 # The 0 here indicates next city is city 0 (start) for path completion

        # If already computed, return the stored result
        if dp[city][mask][0] != float('inf'):
            return dp[city][mask]

        min_cost = float('inf')
        best_next_city = -1

        # Try visiting all unvisited cities
        for next_city in range(n):
            # If next_city is not yet visited (i.e., its bit is not set in mask)
            if not (mask & (1 << next_city)):
                # Calculate cost from current_city to next_city
                # Add it to the minimum cost of traveling from next_city to complete the tour
                cost_from_next, _ = solve(next_city, mask | (1 << next_city))
                current_path_cost = graph[city][next_city] + cost_from_next

                # Update if a cheaper path is found
                if current_path_cost < min_cost:
                    min_cost = current_path_cost
                    best_next_city = next_city

        # Store the computed minimum cost and the best next city for this state
        dp[city][mask] = (min_cost, best_next_city)
        return dp[city][mask]

    # Start the TSP from city 0, with only city 0 visited initially (mask = 1)
    min_total_cost, _ = solve(0, 1)

    # Reconstruct the path
    path = []
    current_city = 0
    mask = 1
    while True:
        path.append(current_city)
        if mask == (1 << n) - 1: # If all cities have been visited
            break

        _, next_city = dp[current_city][mask]
        if next_city == -1: # This means no path was found, or an error in logic
            break
        current_city = next_city
        mask |= (1 << current_city) # Mark the next_city as visited

    path.append(0) # Add the starting city again to complete the cycle

    return min_total_cost, path


In [4]:
graph = [
[0, 10, 15, 20],
[10, 0, 35, 25],
[15, 35, 0, 30],
[20, 25, 30, 0]
]

print("Minimum travelling cost:", tsp(graph))

Minimum travelling cost: (80, [0, 1, 3, 2, 0])


# Exercise
1. Print path of cities visited
2. Take graph input from user
3. Test with different number of cities

In [6]:
# 1. Print path of cities visited (already demonstrated in previous output, but explicitly showing the result here)
min_cost, path = tsp(graph)
print(f"Minimum travelling cost: {min_cost}")
print(f"Optimal path: {path}")

Minimum travelling cost: 80
Optimal path: [0, 1, 3, 2, 0]


In [7]:
# 2. Take graph input from user
def get_user_graph():
    while True:
        try:
            num_cities = int(input("Enter the number of cities (e.g., 4): "))
            if num_cities <= 1:
                print("Number of cities must be greater than 1.")
                continue
            break
        except ValueError:
            print("Invalid input. Please enter an integer.")

    print(f"Enter the distance matrix ({num_cities}x{num_cities}).\nSeparate values by spaces. Enter 0 for distance from a city to itself.")
    print("Example for 3 cities: '0 10 15'\n                   '10 0 20'\n                   '15 20 0'")

    user_graph = []
    for i in range(num_cities):
        while True:
            try:
                row_str = input(f"Enter row {i} (e.g., {' '.join(['0' if i==j else str(10*j) for j in range(num_cities)])}): ")
                row = list(map(int, row_str.split()))
                if len(row) != num_cities:
                    print(f"Row must contain {num_cities} values. Please try again.")
                    continue
                user_graph.append(row)
                break
            except ValueError:
                print("Invalid input. Please enter space-separated integers.")
    return user_graph

user_defined_graph = get_user_graph()
print("\nUser defined graph:")
for row in user_defined_graph:
    print(row)

user_min_cost, user_path = tsp(user_defined_graph)
print(f"\nMinimum travelling cost for user graph: {user_min_cost}")
print(f"Optimal path for user graph: {user_path}")

Enter the number of cities (e.g., 4): 4
Enter the distance matrix (4x4).
Separate values by spaces. Enter 0 for distance from a city to itself.
Example for 3 cities: '0 10 15'
                   '10 0 20'
                   '15 20 0'
Enter row 0 (e.g., 0 10 20 30): 2
Row must contain 4 values. Please try again.
Enter row 0 (e.g., 0 10 20 30): 10,20,30,40
Invalid input. Please enter space-separated integers.
Enter row 0 (e.g., 0 10 20 30): 10 20 30 40 
Enter row 1 (e.g., 0 0 20 30): 0 0 0 0
Enter row 2 (e.g., 0 10 0 30): 2 2 2 2
Enter row 3 (e.g., 0 10 20 0): 4 4 4 4

User defined graph:
[10, 20, 30, 40]
[0, 0, 0, 0]
[2, 2, 2, 2]
[4, 4, 4, 4]

Minimum travelling cost for user graph: 26
Optimal path for user graph: [0, 1, 2, 3, 0]


In [8]:
# 3. Test with different number of cities (e.g., 5 cities)
graph_5_cities = [
    [0, 20, 42, 35, 25],
    [20, 0, 30, 34, 12],
    [42, 30, 0, 10, 6],
    [35, 34, 10, 0, 15],
    [25, 12, 6, 15, 0]
]

print("\nGraph with 5 cities:")
for row in graph_5_cities:
    print(row)

min_cost_5_cities, path_5_cities = tsp(graph_5_cities)
print(f"\nMinimum travelling cost for 5 cities: {min_cost_5_cities}")
print(f"Optimal path for 5 cities: {path_5_cities}")

# Example with 3 cities
graph_3_cities = [
    [0, 10, 15],
    [10, 0, 20],
    [15, 20, 0]
]

print("\nGraph with 3 cities:")
for row in graph_3_cities:
    print(row)

min_cost_3_cities, path_3_cities = tsp(graph_3_cities)
print(f"\nMinimum travelling cost for 3 cities: {min_cost_3_cities}")
print(f"Optimal path for 3 cities: {path_3_cities}")


Graph with 5 cities:
[0, 20, 42, 35, 25]
[20, 0, 30, 34, 12]
[42, 30, 0, 10, 6]
[35, 34, 10, 0, 15]
[25, 12, 6, 15, 0]

Minimum travelling cost for 5 cities: 83
Optimal path for 5 cities: [0, 1, 4, 2, 3, 0]

Graph with 3 cities:
[0, 10, 15]
[10, 0, 20]
[15, 20, 0]

Minimum travelling cost for 3 cities: 45
Optimal path for 3 cities: [0, 1, 2, 0]
